# Conditional DDPM inversion of the Lyα forest

## Flux sightlines → probabilistic baryon-density sightlines

A deterministic neural inverse returns one density field. A conditional denoising diffusion probabilistic model (DDPM) instead learns

$$p(\ln\Delta_b\mid F_{\rm obs}),$$

so one observed Lyα spectrum can produce many plausible density sightlines. From these samples we will calculate:

- a median reconstruction and a pixel-wise 68% posterior interval;
- calibration and field-level reconstruction metrics;
- the density PDF, 1D power spectrum, and folded 1D bispectrum;
- final summary-statistic error bars containing both inter-volume scatter and DDPM posterior uncertainty.

The mock spectra use the simulated H I and temperature fields. The DDPM is used only for the inverse problem.


## 1. Imports and reproducibility

Required packages: `numpy`, `scipy`, `matplotlib`, and `torch`.


In [ ]:
from pathlib import Path
import copy
import math
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, special

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(7)
torch.manual_seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(4, max(1, torch.get_num_threads())))

plt.rcParams.update({
    "figure.figsize": (10.5, 5.0),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

BLACK = "black"
BLUE = "royalblue"
ORANGE = "darkorange"
RED = "crimson"
GREY = "grey"

print(f"PyTorch {torch.__version__}; device = {device}")


## 2. Configuration

`number_of_diffusion_steps = 100` is the requested DDPM chain length. The model is deliberately compact. On a CPU, reduce `training_epochs` or `posterior_samples` for a quick classroom run; on a GPU, the defaults are preferable.


In [ ]:
# Data and cosmology
data_directory = Path("Sims/CMD_z=2_grid128")
unet_checkpoint_path = Path("trained_models/lya_forest_unet_density.pt")
ddpm_checkpoint_path = Path("trained_models/lya_forest_conditional_ddpm.pt")
box_size = 25.0                 # h^-1 cMpc
box_redshift = 2.0
h_camels = 0.6711
Omega_m = 0.30

# Mock observation
signal_to_noise = 30.0
instrument_fwhm_kms = 50.0

# Dataset sizes
training_skewers_per_box = 96
validation_skewers_per_box = 64
test_skewers_per_box = 16

# Conditional DDPM
number_of_diffusion_steps = 100
# Change these two values to alter both the encoder and mirrored decoder.
number_of_encoder_decoder_layers = 2   # try 1, 2, 3, or 4
convolution_kernel_size = 5            # positive odd integer: 1, 3, 5, 7, ...
base_channels = 16
time_embedding_size = 64
use_fully_connected_bottleneck = False
bottleneck_latent_size = 64

# Optimisation and posterior sampling
training_epochs = 100
batch_size = 64
learning_rate = 2.0e-4
posterior_samples = 32         # enough for useful 16th/84th percentiles
sampling_sightline_batch = 16  # controls sampling memory

# Common resolution used for quantitative comparisons
comparison_fwhm = 1.0          # h^-1 cMpc


## 3. Load CAMELS fields

Each complete simulation box belongs to only one split. This is more stringent than randomly splitting sightlines from the same boxes.


In [ ]:
gas_path = data_directory / "Grids_Mgas_IllustrisTNG_CV_128_z=2.0.npy"
hi_path = data_directory / "Grids_HI_IllustrisTNG_CV_128_z=2.0.npy"
temperature_path = data_directory / "Grids_T_IllustrisTNG_CV_128_z=2.0.npy"

missing_paths = [
    path for path in (gas_path, hi_path, temperature_path) if not path.exists()
]
if missing_paths:
    missing_names = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "The CAMELS grids are required. Missing:\n" + missing_names
    )

gas_boxes = np.load(gas_path, mmap_mode="r")
hi_boxes = np.load(hi_path, mmap_mode="r")
temperature_boxes = np.load(temperature_path, mmap_mode="r")

number_of_cells = gas_boxes.shape[1]
cell_size = box_size / number_of_cells
x = (np.arange(number_of_cells) + 0.5) * cell_size

training_simulations = np.arange(0, 18)
validation_simulations = np.arange(18, 20)
test_simulations = np.arange(20, 27)

assert gas_boxes.shape[0] >= 27
assert gas_boxes.shape == hi_boxes.shape == temperature_boxes.shape

gas_means = np.array([
    gas_boxes[index].mean(dtype=np.float64)
    for index in range(gas_boxes.shape[0])
])

print(f"Grid shape: {gas_boxes.shape}")
print(f"Cell width: {cell_size:.3f} h^-1 cMpc")
print("Training / validation / test boxes: "
      f"{len(training_simulations)} / {len(validation_simulations)} / "
      f"{len(test_simulations)}")


## 4. Realistic Lyα forward model

For each absorber cell, the H I density fixes the optical-depth amplitude and the temperature fixes the Voigt-profile width. We then apply instrumental smoothing and Gaussian pixel noise:

$$F_{\rm obs}=\mathcal{G}_{\rm inst}*\exp(-\tau_{\rm Voigt})+\epsilon_F.$$

Peculiar velocities are omitted, as in the supplied deterministic-inversion notebook.


In [ ]:
# Physical constants in cgs units
MSUN_G = 1.98847e33
MPC_CM = 3.085677581e24
M_H_G = 1.6735575e-24
K_B = 1.380649e-16
C_CMS = 2.99792458e10
C_KMS = C_CMS / 1.0e5

# Ly-alpha atomic constants
LAMBDA_ALPHA_CM = 1215.67e-8
GAMMA_ALPHA = 6.262e8
I_ALPHA = 4.45e-18

H_z = 100.0 * h_camels * np.sqrt(
    Omega_m * (1.0 + box_redshift)**3 + (1.0 - Omega_m)
)
distance_from_centre_Mpc = (x - box_size / 2.0) / h_camels
absorber_redshift = (
    box_redshift + H_z * distance_from_centre_Mpc / C_KMS
)


def hi_grid_to_number_density(rho_hi_grid):
    '''Convert a CAMELS H I grid to physical H I number density [cm^-3].'''
    rho_comoving = rho_hi_grid * MSUN_G * h_camels**2 / MPC_CM**3
    rho_physical = rho_comoving * (1.0 + absorber_redshift)**3
    return rho_physical / M_H_G


def continuous_voigt_optical_depth(rho_hi_grid, temperature):
    '''Optical depth from cell-sampled H I and temperature fields.'''
    n_hi = hi_grid_to_number_density(rho_hi_grid)
    nu_alpha = C_CMS / LAMBDA_ALPHA_CM
    cell_width_cm = cell_size * MPC_CM / h_camels

    safe_temperature = np.clip(temperature, 10.0, None)
    b_cms = np.sqrt(2.0 * K_B * safe_temperature / M_H_G)
    damping = GAMMA_ALPHA * C_CMS / (4.0 * np.pi * nu_alpha * b_cms)

    velocity_offset = (
        C_CMS
        * (absorber_redshift[:, None] - absorber_redshift[None, :])
        / (1.0 + absorber_redshift[None, :])
    )
    profile = np.real(special.wofz(
        velocity_offset / b_cms[:, None] + 1j * damping[:, None]
    ))
    amplitude = (
        C_CMS * I_ALPHA * cell_width_cm * n_hi
        / (np.sqrt(np.pi) * b_cms * (1.0 + absorber_redshift))
    )
    return np.sum(amplitude[:, None] * profile, axis=0)


velocity_pixel_width = H_z * (cell_size / h_camels) / (1.0 + box_redshift)
instrument_sigma_pixels = (
    instrument_fwhm_kms
    / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / velocity_pixel_width
)
noise_sigma = 1.0 / signal_to_noise


def observe_skewer(rho_hi, temperature, noise_rng):
    '''Return intrinsic, instrumentally smoothed, and noisy flux.'''
    optical_depth = continuous_voigt_optical_depth(rho_hi, temperature)
    intrinsic_flux = np.exp(-optical_depth)
    instrumental_flux = ndimage.gaussian_filter1d(
        intrinsic_flux, instrument_sigma_pixels, mode="wrap"
    )
    observed_flux = instrumental_flux + noise_rng.normal(
        0.0, noise_sigma, number_of_cells
    )
    return intrinsic_flux, instrumental_flux, observed_flux


print(f"Instrument FWHM = {instrument_fwhm_kms:.1f} km/s")
print(f"S/N = {signal_to_noise:.1f}; sigma_F = {noise_sigma:.4f}")


## 5. Generate paired sightlines

The condition is the noisy observed flux $F_{\rm obs}(x)$. The clean target is

$$s_0(x)=\frac{\ln\Delta_b(x)-\mu_s}{\sigma_s}.$$


In [ ]:
def choose_positions(number, rng, include_centre=False):
    '''Choose unique transverse coordinates in one simulation box.'''
    selected = []
    if include_centre:
        selected.append((number_of_cells // 2, number_of_cells // 2))

    centre_flat = (number_of_cells // 2) * number_of_cells + number_of_cells // 2
    available = np.delete(np.arange(number_of_cells**2), centre_flat)
    random_flat = rng.choice(
        available, number - len(selected), replace=False
    )
    selected.extend(divmod(int(index), number_of_cells) for index in random_flat)
    return selected


def build_dataset(simulations, skewers_per_box, seed, include_centre=False):
    '''Generate density targets and realistic noisy flux conditions.'''
    position_rng = np.random.default_rng(seed)
    noise_rng = np.random.default_rng(seed + 1)

    densities, intrinsic_fluxes, observed_fluxes, labels = [], [], [], []

    for simulation in simulations:
        positions = choose_positions(
            skewers_per_box, position_rng, include_centre=include_centre
        )
        for y_index, z_index in positions:
            density = (
                gas_boxes[simulation, :, y_index, z_index].astype(float)
                / gas_means[simulation]
            )
            rho_hi = hi_boxes[simulation, :, y_index, z_index].astype(float)
            temperature = temperature_boxes[
                simulation, :, y_index, z_index
            ].astype(float)

            intrinsic, _, observed = observe_skewer(
                rho_hi, temperature, noise_rng
            )
            densities.append(density)
            intrinsic_fluxes.append(intrinsic)
            observed_fluxes.append(observed)
            labels.append(simulation)

    return {
        "density": np.asarray(densities, dtype=np.float32),
        "intrinsic_flux": np.asarray(intrinsic_fluxes, dtype=np.float32),
        "observed_flux": np.asarray(observed_fluxes, dtype=np.float32),
        "simulation": np.asarray(labels),
    }


generation_start = time.perf_counter()
train_data = build_dataset(
    training_simulations, training_skewers_per_box, seed=101
)
validation_data = build_dataset(
    validation_simulations, validation_skewers_per_box, seed=202
)
test_data = build_dataset(
    test_simulations, test_skewers_per_box, seed=303, include_centre=True
)

print(f"Training pairs:   {train_data['density'].shape}")
print(f"Validation pairs: {validation_data['density'].shape}")
print(f"Test pairs:       {test_data['density'].shape}")
print(f"Pair generation time: {time.perf_counter()-generation_start:.1f} s")


In [ ]:
representative = 0
fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.0), sharex=True,
                         constrained_layout=True)
axes[0].plot(x, test_data["observed_flux"][representative],
             color=BLUE, lw=1.0)
axes[0].set(ylabel=r"$F_{\rm obs}$", title="One held-out mock sightline")
axes[0].axhline(0.0, color=GREY, lw=0.7)
axes[1].plot(x, test_data["density"][representative],
             color=BLACK, lw=1.3)
axes[1].set(xlabel=r"$x$ [$h^{-1}$ cMpc]", ylabel=r"$\Delta_b$")
axes[1].set_yscale("log")
for ax in axes:
    ax.grid(alpha=0.18)
plt.show()


## 6. Standardize using the training set only

We diffuse standardized log-density rather than raw density. This keeps the target closer to a well-scaled, approximately symmetric variable while guaranteeing positive physical densities after exponentiation.


In [ ]:
flux_mean = float(train_data["observed_flux"].mean())
flux_std = float(train_data["observed_flux"].std())

train_log_density = np.log(np.clip(train_data["density"], 1.0e-4, None))
validation_log_density = np.log(
    np.clip(validation_data["density"], 1.0e-4, None)
)
target_mean = float(train_log_density.mean())
target_std = float(train_log_density.std())


def flux_tensor(flux):
    standardized = (flux - flux_mean) / flux_std
    return torch.as_tensor(standardized[:, None, :], dtype=torch.float32)


def target_tensor(log_density):
    standardized = (log_density - target_mean) / target_std
    return torch.as_tensor(standardized[:, None, :], dtype=torch.float32)


training_set = TensorDataset(
    flux_tensor(train_data["observed_flux"]),
    target_tensor(train_log_density),
)
validation_set = TensorDataset(
    flux_tensor(validation_data["observed_flux"]),
    target_tensor(validation_log_density),
)

loader_generator = torch.Generator().manual_seed(7)
training_loader = DataLoader(
    training_set, batch_size=batch_size, shuffle=True,
    generator=loader_generator, num_workers=0
)
validation_loader = DataLoader(
    validation_set, batch_size=2*batch_size, shuffle=False, num_workers=0
)

example_flux, example_target = next(iter(training_loader))
print("Condition batch:", tuple(example_flux.shape))
print("Target batch:   ", tuple(example_target.shape))


## 7. DDPM forward process: add noise in 100 steps

We use a cosine schedule so that a short 100-step chain still ends very close to pure Gaussian noise. With $\alpha_t=1-\beta_t$ and $\bar\alpha_t=\prod_{j=1}^{t}\alpha_j$, a clean target $s_0$ can be noised directly to any time $t$:

$$s_t=\sqrt{\bar\alpha_t}\,s_0+\sqrt{1-\bar\alpha_t}\,\epsilon,
\qquad \epsilon\sim\mathcal N(0,I).$$

Training therefore needs only one randomly selected time step per example. The flux condition is never noised.


In [ ]:
def cosine_beta_schedule(number_of_steps, offset=0.008):
    '''Cosine schedule, rescaled to reach almost pure noise at the end.'''
    steps = torch.arange(number_of_steps + 1, dtype=torch.float64)
    phase = ((steps / number_of_steps) + offset) / (1.0 + offset)
    alpha_bar_curve = torch.cos(0.5 * np.pi * phase)**2
    alpha_bar_curve = alpha_bar_curve / alpha_bar_curve[0]
    beta_values = 1.0 - alpha_bar_curve[1:] / alpha_bar_curve[:-1]
    return beta_values.clamp(1.0e-5, 0.999).float()


betas = cosine_beta_schedule(number_of_diffusion_steps).to(device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
alpha_bars_previous = torch.cat([
    torch.ones(1, device=device), alpha_bars[:-1]
])

sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
posterior_variance = (
    betas * (1.0 - alpha_bars_previous) / (1.0 - alpha_bars)
).clamp(min=1.0e-20)


def extract(values, timesteps, target):
    '''Select one schedule value per batch item and add spatial axes.'''
    return values[timesteps].reshape(-1, *([1] * (target.ndim - 1)))


def q_sample(clean_target, timesteps, noise=None):
    '''Draw s_t directly from q(s_t | s_0).'''
    if noise is None:
        noise = torch.randn_like(clean_target)
    return (
        extract(sqrt_alpha_bars, timesteps, clean_target) * clean_target
        + extract(sqrt_one_minus_alpha_bars, timesteps, clean_target) * noise
    )


In [ ]:
clean_example = example_target[:1].to(device)
shared_noise = torch.randn_like(clean_example)
shown_steps = [0, 24, 49, 74, 99]

fig, axes = plt.subplots(len(shown_steps), 1, figsize=(10.5, 8.0),
                         sharex=True, constrained_layout=True)
for ax, step in zip(axes, shown_steps):
    t = torch.tensor([step], device=device)
    noisy = q_sample(clean_example, t, shared_noise)[0, 0].cpu().numpy()
    ax.plot(x, noisy, color=ORANGE, lw=1.0)
    ax.set_ylabel(fr"$s_{{{step+1}}}$")
    ax.grid(alpha=0.15)
axes[-1].set_xlabel(r"$x$ [$h^{-1}$ cMpc]")
fig.suptitle("Forward diffusion of one standardized log-density sightline")
plt.show()


## 8. Conditional noise-prediction network

At each reverse step, the network receives three pieces of information:

$$\epsilon_\theta\left(s_t,\,F_{\rm obs},\,t\right).$$

A small 1D U-Net combines the current noisy density with the observed flux. A sinusoidal embedding tells every convolutional block how much noise is present. Circular padding respects the periodic simulation sightline.

Both sides of the U-Net are controlled from the configuration cell:

- `number_of_encoder_decoder_layers` changes the number of downsampling stages and creates the same number of mirrored upsampling stages;
- `convolution_kernel_size` changes the kernel width in every convolutional block;
- `use_fully_connected_bottleneck` optionally compresses the DDPM U-Net bottleneck to `bottleneck_latent_size` features before expanding it again.


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.embedding_size = embedding_size
        half = self.embedding_size // 2
        frequencies = torch.exp(
            -math.log(10_000.0)
            * torch.arange(half)
            / max(half - 1, 1)
        )
        self.register_buffer("frequencies", frequencies)

    def forward(self, timesteps):
        angles = (
            timesteps.float()[:, None] * self.frequencies[None, :]
        )
        embedding = torch.cat([angles.sin(), angles.cos()], dim=1)
        if self.embedding_size % 2 == 1:
            embedding = F.pad(embedding, (0, 1))
        return embedding


class TimeConditionedBlock(nn.Module):
    def __init__(self, input_channels, output_channels, time_size, kernel_size):
        super().__init__()
        if kernel_size < 1 or kernel_size % 2 == 0:
            raise ValueError("kernel_size must be a positive odd integer")
        padding = kernel_size // 2

        self.conv1 = nn.Conv1d(
            input_channels, output_channels, kernel_size,
            padding=padding, padding_mode="circular"
        )
        self.conv2 = nn.Conv1d(
            output_channels, output_channels, kernel_size,
            padding=padding, padding_mode="circular"
        )
        self.time_projection = nn.Linear(time_size, output_channels)
        self.norm1 = nn.GroupNorm(1, output_channels)
        self.norm2 = nn.GroupNorm(1, output_channels)
        self.residual = (
            nn.Identity() if input_channels == output_channels
            else nn.Conv1d(input_channels, output_channels, kernel_size=1)
        )

    def forward(self, values, time_embedding):
        hidden = self.conv1(values)
        hidden = hidden + self.time_projection(time_embedding)[:, :, None]
        hidden = F.silu(self.norm1(hidden))
        hidden = self.conv2(hidden)
        hidden = self.norm2(hidden)
        return F.silu(hidden + self.residual(values))


class ConditionalUNet1D(nn.Module):
    def __init__(
        self,
        number_of_layers=2,
        base_channels=16,
        time_size=64,
        kernel_size=5,
        input_length=128,
        use_fully_connected_bottleneck=False,
        bottleneck_latent_size=128,
    ):
        super().__init__()
        if number_of_layers < 1:
            raise ValueError("number_of_layers must be at least 1")
        downsample_factor = 2**number_of_layers
        if input_length % downsample_factor != 0:
            raise ValueError(
                "input_length must be divisible by 2**number_of_layers"
            )
        if use_fully_connected_bottleneck and bottleneck_latent_size < 1:
            raise ValueError("bottleneck_latent_size must be positive")
        self.number_of_layers = number_of_layers
        self.use_fully_connected_bottleneck = (
            use_fully_connected_bottleneck
        )
        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_size),
            nn.Linear(time_size, time_size),
            nn.SiLU(),
            nn.Linear(time_size, time_size),
        )
        self.stem = nn.Conv1d(
            2, base_channels, kernel_size,
            padding=kernel_size // 2, padding_mode="circular"
        )

        # Encoder: double the channel count after every downsampling stage.
        self.down_blocks = nn.ModuleList()
        current_channels = base_channels
        for layer_index in range(number_of_layers):
            output_channels = base_channels * 2**(layer_index + 1)
            self.down_blocks.append(TimeConditionedBlock(
                current_channels, output_channels, time_size, kernel_size
            ))
            current_channels = output_channels

        self.middle = TimeConditionedBlock(
            current_channels, current_channels, time_size, kernel_size
        )
        self.bottleneck_channels = current_channels
        self.bottleneck_length = input_length // downsample_factor
        if use_fully_connected_bottleneck:
            flattened_size = (
                self.bottleneck_channels * self.bottleneck_length
            )
            self.fully_connected_bottleneck = nn.Sequential(
                nn.Linear(flattened_size, bottleneck_latent_size),
                nn.SiLU(),
                nn.Linear(bottleneck_latent_size, flattened_size),
            )
        else:
            self.fully_connected_bottleneck = None

        # Decoder: mirror the encoder and concatenate the skip features.
        self.up_blocks = nn.ModuleList()
        for layer_index in reversed(range(number_of_layers)):
            skip_channels = base_channels * 2**(layer_index + 1)
            output_channels = base_channels * 2**layer_index
            self.up_blocks.append(TimeConditionedBlock(
                current_channels + skip_channels,
                output_channels,
                time_size,
                kernel_size,
            ))
            current_channels = output_channels

        self.output = nn.Conv1d(current_channels, 1, kernel_size=1)

    def forward(self, noisy_density, observed_flux, timesteps):
        time_embedding = self.time_embedding(timesteps)
        values = self.stem(torch.cat([noisy_density, observed_flux], dim=1))

        skip_features = []
        for block in self.down_blocks:
            values = block(values, time_embedding)
            skip_features.append(values)
            values = F.avg_pool1d(values, 2)

        values = self.middle(values, time_embedding)
        if self.use_fully_connected_bottleneck:
            batch_size = values.shape[0]
            values = self.fully_connected_bottleneck(values.flatten(1))
            values = values.reshape(
                batch_size, self.bottleneck_channels,
                self.bottleneck_length,
            )

        for block, skip_feature in zip(
            self.up_blocks, reversed(skip_features)
        ):
            values = F.interpolate(
                values,
                size=skip_feature.shape[-1],
                mode="linear",
                align_corners=False,
            )
            values = block(
                torch.cat([values, skip_feature], dim=1),
                time_embedding,
            )

        return self.output(values)


model = ConditionalUNet1D(
    number_of_layers=number_of_encoder_decoder_layers,
    base_channels=base_channels,
    time_size=time_embedding_size,
    kernel_size=convolution_kernel_size,
    input_length=number_of_cells,
    use_fully_connected_bottleneck=use_fully_connected_bottleneck,
    bottleneck_latent_size=bottleneck_latent_size,
).to(device)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
with torch.no_grad():
    test_t = torch.zeros(example_target[:2].shape[0], dtype=torch.long,
                         device=device)
    test_output = model(
        example_target[:2].to(device), example_flux[:2].to(device), test_t
    )

print("Model output shape:", tuple(test_output.shape))
print(f"Encoder/decoder layers: {number_of_encoder_decoder_layers}")
print(f"Convolution kernel size: {convolution_kernel_size}")
print(f"Fully connected bottleneck: {use_fully_connected_bottleneck}")
if use_fully_connected_bottleneck:
    print(f"Bottleneck latent size: {bottleneck_latent_size}")
print(f"Trainable parameters: {parameter_count:,}")


## 9. Train by predicting the added noise

For every mini-batch we draw a random $t$ and random $\epsilon$, build $s_t$, and minimize

$$\mathcal L=\left\|\epsilon-\epsilon_\theta(s_t,F_{\rm obs},t)\right\|_2^2.$$

The best validation checkpoint is retained. The validation random numbers are reset each epoch, so its curve is directly comparable across epochs. The retained model and its inference metadata are saved to `trained_models/lya_forest_conditional_ddpm.pt`.


In [ ]:
def run_validation(model):
    model.eval()
    validation_sum = 0.0
    validation_generator = torch.Generator(device=device).manual_seed(909)

    with torch.no_grad():
        for condition, clean_target in validation_loader:
            condition = condition.to(device)
            clean_target = clean_target.to(device)
            timesteps = torch.randint(
                0, number_of_diffusion_steps, (condition.size(0),),
                generator=validation_generator, device=device
            )
            noise = torch.randn(
                clean_target.shape, generator=validation_generator, device=device
            )
            noisy_target = q_sample(clean_target, timesteps, noise)
            predicted_noise = model(noisy_target, condition, timesteps)
            validation_sum += (
                F.mse_loss(predicted_noise, noise).item() * condition.size(0)
            )

    return validation_sum / len(validation_set)


def train_ddpm(model):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=1.0e-4
    )
    history = {"training": [], "validation": []}
    best_validation = np.inf
    best_state = None
    start = time.perf_counter()

    for epoch in range(1, training_epochs + 1):
        model.train()
        training_sum = 0.0

        for condition, clean_target in training_loader:
            condition = condition.to(device)
            clean_target = clean_target.to(device)
            timesteps = torch.randint(
                0, number_of_diffusion_steps, (condition.size(0),), device=device
            )
            noise = torch.randn_like(clean_target)
            noisy_target = q_sample(clean_target, timesteps, noise)

            optimizer.zero_grad(set_to_none=True)
            predicted_noise = model(noisy_target, condition, timesteps)
            loss = F.mse_loss(predicted_noise, noise)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            training_sum += loss.item() * condition.size(0)

        training_loss = training_sum / len(training_set)
        validation_loss = run_validation(model)
        history["training"].append(training_loss)
        history["validation"].append(validation_loss)

        if validation_loss < best_validation:
            best_validation = validation_loss
            best_state = copy.deepcopy(model.state_dict())

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"epoch {epoch:3d}/{training_epochs}: "
                f"train={training_loss:.4f}, validation={validation_loss:.4f}"
            )

    model.load_state_dict(best_state)
    print(f"Retained best model; elapsed time = {time.perf_counter()-start:.1f} s")
    return history


In [ ]:
history = train_ddpm(model)


class SavedConditionalDDPM(nn.Module):
    '''Bundle the trained noise predictor and inference metadata.'''
    def __init__(self, network):
        super().__init__()
        self.network = network
        self.register_buffer(
            "flux_mean", torch.tensor(flux_mean, dtype=torch.float32)
        )
        self.register_buffer(
            "flux_std", torch.tensor(flux_std, dtype=torch.float32)
        )
        self.register_buffer(
            "target_mean", torch.tensor(target_mean, dtype=torch.float32)
        )
        self.register_buffer(
            "target_std", torch.tensor(target_std, dtype=torch.float32)
        )
        self.register_buffer("betas", betas.detach().clone())
        self.register_buffer("alphas", alphas.detach().clone())
        self.register_buffer("alpha_bars", alpha_bars.detach().clone())
        self.register_buffer(
            "alpha_bars_previous", alpha_bars_previous.detach().clone()
        )
        self.register_buffer(
            "sqrt_one_minus_alpha_bars",
            sqrt_one_minus_alpha_bars.detach().clone(),
        )
        self.register_buffer(
            "posterior_variance", posterior_variance.detach().clone()
        )

    def forward(self, noisy_density, observed_flux, timesteps):
        standardized_flux = (observed_flux - self.flux_mean) / self.flux_std
        return self.network(
            noisy_density, standardized_flux[:, None, :], timesteps
        )


model.eval()
saved_ddpm = SavedConditionalDDPM(model).to(device).eval()
trace_noisy_density = example_target[:2].to(device)
trace_observed_flux = torch.as_tensor(
    train_data["observed_flux"][:2], dtype=torch.float32, device=device
)
trace_timesteps = torch.zeros(2, dtype=torch.long, device=device)
traced_ddpm = torch.jit.trace(
    saved_ddpm,
    (trace_noisy_density, trace_observed_flux, trace_timesteps),
)
ddpm_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.jit.save(traced_ddpm, str(ddpm_checkpoint_path))
print(f"Saved trained conditional DDPM to {ddpm_checkpoint_path}")

epochs = np.arange(1, training_epochs + 1)
fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
ax.plot(epochs, history["training"], color=BLUE, label="training")
ax.plot(epochs, history["validation"], color=ORANGE, label="validation")
ax.set(xlabel="epoch", ylabel="noise-prediction MSE",
       title="Conditional DDPM training")
ax.set_yscale("log")
ax.grid(alpha=0.18)
ax.legend()
plt.show()


## 10. Reverse diffusion: sample density conditioned on flux

Sampling starts from Gaussian noise, $s_{100}\sim\mathcal N(0,I)$, and applies all 100 reverse steps while keeping the same observed flux fixed. Different initial noise and reverse-process noise produce different physically plausible inversions.

The network predicts $s_0$ indirectly through its noise estimate. We clip only the standardized $s_0$ estimate to a broad range to prevent rare numerical excursions during a short educational training run.


In [ ]:
@torch.no_grad()
def reverse_step(model, noisy_target, condition, step, generator):
    '''One ancestral DDPM step p_theta(s_{t-1} | s_t, condition).'''
    timesteps = torch.full(
        (noisy_target.size(0),), step, dtype=torch.long, device=device
    )
    predicted_noise = model(noisy_target, condition, timesteps)

    alpha_bar = extract(alpha_bars, timesteps, noisy_target)
    predicted_clean = (
        noisy_target
        - extract(sqrt_one_minus_alpha_bars, timesteps, noisy_target)
        * predicted_noise
    ) / torch.sqrt(alpha_bar)
    predicted_clean = predicted_clean.clamp(-5.0, 5.0)

    alpha = extract(alphas, timesteps, noisy_target)
    beta = extract(betas, timesteps, noisy_target)
    alpha_bar_previous = extract(
        alpha_bars_previous, timesteps, noisy_target
    )
    denominator = 1.0 - alpha_bar
    posterior_mean = (
        beta * torch.sqrt(alpha_bar_previous) / denominator * predicted_clean
        + (1.0 - alpha_bar_previous) * torch.sqrt(alpha)
        / denominator * noisy_target
    )

    if step == 0:
        return posterior_mean

    noise = torch.randn(
        noisy_target.shape, generator=generator, device=device
    )
    variance = extract(posterior_variance, timesteps, noisy_target)
    return posterior_mean + torch.sqrt(variance) * noise


@torch.no_grad()
def sample_density(model, observed_flux, number_of_samples, seed=1234):
    '''Return physical density samples with shape [sample, sightline, pixel].'''
    model.eval()
    all_chunks = []
    generator = torch.Generator(device=device).manual_seed(seed)
    conditions = flux_tensor(observed_flux)

    for start in range(0, len(conditions), sampling_sightline_batch):
        condition = conditions[start:start+sampling_sightline_batch].to(device)
        n_sightlines = condition.size(0)
        condition = condition.repeat_interleave(number_of_samples, dim=0)

        noisy_target = torch.randn(
            condition.shape, generator=generator, device=device
        )
        for step in reversed(range(number_of_diffusion_steps)):
            noisy_target = reverse_step(
                model, noisy_target, condition, step, generator
            )

        standardized = noisy_target[:, 0].reshape(
            n_sightlines, number_of_samples, number_of_cells
        ).permute(1, 0, 2)
        all_chunks.append(standardized.cpu().numpy())

    standardized_samples = np.concatenate(all_chunks, axis=1)
    log_density_samples = target_mean + target_std * standardized_samples
    return np.exp(np.clip(log_density_samples, np.log(1.0e-4), np.log(1.0e3)))


sampling_start = time.perf_counter()
ddpm_density_samples = sample_density(
    model, test_data["observed_flux"], posterior_samples
)
print("Posterior sample array:", ddpm_density_samples.shape)
print(f"Sampling time: {time.perf_counter()-sampling_start:.1f} s")


## 11. Load the saved U-Net and compare sightline predictions

The deterministic U-Net is loaded from the checkpoint written by `Lya_forest_neural_vs_FGPA_improved.ipynb`; it is not trained again here. The saved inference model includes the U-Net's training-set normalization. For every DDPM sightline pixel we use the 16th, 50th, and 84th percentiles of the samples, giving a central 68% posterior credible interval.


In [ ]:
if not unet_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Missing {unet_checkpoint_path}. Run "
        "Lya_forest_neural_vs_FGPA_improved.ipynb first."
    )

saved_unet = torch.jit.load(str(unet_checkpoint_path), map_location=device)
saved_unet.eval()


@torch.no_grad()
def predict_saved_unet(observed_flux, prediction_batch_size=256):
    '''Predict physical gas overdensity without retraining the U-Net.'''
    predictions = []
    for start in range(0, len(observed_flux), prediction_batch_size):
        stop = min(start + prediction_batch_size, len(observed_flux))
        input_batch = torch.as_tensor(
            observed_flux[start:stop], dtype=torch.float32, device=device
        )
        predictions.append(saved_unet(input_batch).cpu().numpy())
    return np.concatenate(predictions, axis=0)


unet_density_test = predict_saved_unet(test_data["observed_flux"])
print(f"Loaded saved U-Net from {unet_checkpoint_path}")

ddpm_p16, ddpm_median, ddpm_p84 = np.percentile(
    ddpm_density_samples, [16, 50, 84], axis=0
)

fig, axes = plt.subplots(2, 1, figsize=(11, 7.0), sharex=True,
                         constrained_layout=True)
axes[0].plot(x, test_data["observed_flux"][representative],
             color=BLUE, lw=1.0)
axes[0].set(ylabel=r"$F_{\rm obs}$", title="Saved U-Net and conditional DDPM inversion")
axes[0].axhline(0.0, color=GREY, lw=0.7)

# A few individual posterior samples make the stochastic prediction visible.
for sample in ddpm_density_samples[:8, representative]:
    axes[1].plot(x, sample, color=ORANGE, alpha=0.10, lw=0.7)
axes[1].fill_between(
    x, ddpm_p16[representative], ddpm_p84[representative],
    color=ORANGE, alpha=0.28, label="DDPM central 68% interval"
)
axes[1].plot(x, ddpm_median[representative], color=ORANGE, lw=1.5,
             label="DDPM median")
axes[1].plot(x, unet_density_test[representative], color=RED, lw=1.4,
             label="saved U-Net")
axes[1].plot(x, test_data["density"][representative], color=BLACK,
             lw=1.2, label="truth")
axes[1].set(xlabel=r"$x$ [$h^{-1}$ cMpc]", ylabel=r"$\Delta_b$")
axes[1].set_yscale("log")
axes[1].legend(ncol=3, fontsize=9)
for ax in axes:
    ax.grid(alpha=0.18)
plt.show()


## 12. Calibration and field-level performance

If the posterior were perfectly calibrated, the truth would fall inside a nominal 68% interval in roughly 68% of test pixels. Coverage diagnoses uncertainty calibration; it does not measure reconstruction accuracy. We calculate RMSE, bias, and correlation for both the DDPM posterior median and the saved U-Net after smoothing truth and predictions to the same scale.


In [ ]:
raw_coverage = np.mean(
    (test_data["density"] >= ddpm_p16)
    & (test_data["density"] <= ddpm_p84)
)

comparison_sigma_pixels = (
    comparison_fwhm
    / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / cell_size
)


def smooth_fields(fields):
    return ndimage.gaussian_filter1d(
        fields, comparison_sigma_pixels, axis=-1, mode="wrap"
    )


truth_matched = smooth_fields(test_data["density"])
median_matched = smooth_fields(ddpm_median)
samples_matched = smooth_fields(ddpm_density_samples)
unet_matched = smooth_fields(unet_density_test)


def field_metrics_per_skewer(truth, prediction):
    log_truth = np.log10(np.clip(truth, 1.0e-4, None))
    log_prediction = np.log10(np.clip(prediction, 1.0e-4, None))
    difference = log_prediction - log_truth
    return {
        "RMSE [dex]": np.sqrt(np.mean(difference**2, axis=1)),
        "bias [dex]": np.mean(difference, axis=1),
        "correlation": np.array([
            np.corrcoef(true_row, predicted_row)[0, 1]
            for true_row, predicted_row in zip(log_truth, log_prediction)
        ]),
    }


def average_within_each_test_simulation(values):
    return np.array([
        np.mean(values[test_data["simulation"] == simulation], axis=0)
        for simulation in test_simulations
    ])


field_metric_samples = {}
for method, prediction in {
    "saved U-Net": unet_matched,
    "DDPM median": median_matched,
}.items():
    field_metric_samples[method] = {
        name: average_within_each_test_simulation(values)
        for name, values in field_metrics_per_skewer(
            truth_matched, prediction
        ).items()
    }

print(f"Raw-pixel DDPM 68% interval coverage: {100*raw_coverage:.1f}%")
print(f"Metrics at {comparison_fwhm:.1f} h^-1 cMpc FWHM")
print(f"{'Method':14s} {'RMSE [dex]':>20s} {'bias [dex]':>20s} "
      f"{'correlation':>20s}")
for method, samples in field_metric_samples.items():
    print(
        f"{method:14s} "
        f"{samples['RMSE [dex]'].mean():7.3f} +/- "
        f"{samples['RMSE [dex]'].std(ddof=1):.3f} "
        f"{samples['bias [dex]'].mean():+7.3f} +/- "
        f"{samples['bias [dex]'].std(ddof=1):.3f} "
        f"{samples['correlation'].mean():7.3f} +/- "
        f"{samples['correlation'].std(ddof=1):.3f}"
    )


## 13. Summary statistics in each held-out volume

We calculate each statistic separately for every sightline and then average sightlines belonging to the same held-out simulation. Thus the first axis of each result is the seven independent test volumes.

For the DDPM, this operation is repeated for every posterior draw, producing arrays with shape

$$[N_{\rm sample},\,N_{\rm volume},\,N_{\rm bin}].$$

The saved U-Net is evaluated once on the identical held-out sightlines, so its PDF, power spectrum, and folded bispectrum can be compared directly with the DDPM summaries.


In [ ]:
pdf_edges = np.linspace(-2.0, 2.0, 33)
pdf_centres = 0.5 * (pdf_edges[:-1] + pdf_edges[1:])


def density_pdf_per_skewer(density):
    return np.asarray([
        np.histogram(
            np.log10(np.clip(row, 1.0e-4, None)),
            bins=pdf_edges, density=True
        )[0]
        for row in density
    ])


def power_per_skewer(density):
    contrast = density / density.mean(axis=1, keepdims=True) - 1.0
    transform = cell_size * np.fft.rfft(contrast, axis=1)
    k = 2.0 * np.pi * np.fft.rfftfreq(number_of_cells, d=cell_size)
    power = np.abs(transform)**2 / box_size
    return k[1:], power[:, 1:]


def folded_bispectrum_per_skewer(density):
    contrast = density / density.mean(axis=1, keepdims=True) - 1.0
    transform = cell_size * np.fft.rfft(contrast, axis=1)
    modes = np.arange(1, number_of_cells // 4)
    k = 2.0 * np.pi * modes / box_size
    bispectrum = np.real(
        transform[:, modes]**2 * np.conj(transform[:, 2*modes])
    ) / box_size
    return k, bispectrum


def bin_modes(k, statistic, number_of_bins):
    edges = np.logspace(np.log10(k.min()), np.log10(k.max()),
                        number_of_bins + 1)
    centres = np.sqrt(edges[:-1] * edges[1:])
    binned = np.full((statistic.shape[0], number_of_bins), np.nan)
    for index in range(number_of_bins):
        in_bin = (k >= edges[index]) & (k < edges[index+1])
        if index == number_of_bins - 1:
            in_bin = (k >= edges[index]) & (k <= edges[index+1])
        if np.any(in_bin):
            binned[:, index] = statistic[:, in_bin].mean(axis=1)
    return centres, binned


def summaries_by_volume(fields):
    '''Return PDF, P(k), and B(k,k,-2k) for each held-out volume.'''
    pdf_skewers = density_pdf_per_skewer(fields)
    power_k_raw, power_raw = power_per_skewer(fields)
    power_k, power_skewers = bin_modes(power_k_raw, power_raw, 10)
    bispectrum_k_raw, bispectrum_raw = folded_bispectrum_per_skewer(fields)
    bispectrum_k, bispectrum_skewers = bin_modes(
        bispectrum_k_raw, bispectrum_raw, 8
    )
    return {
        "PDF": average_within_each_test_simulation(pdf_skewers),
        "power": average_within_each_test_simulation(power_skewers),
        "bispectrum": average_within_each_test_simulation(bispectrum_skewers),
    }


# The k-bin centres depend only on the grid, so calculate them once for plotting.
power_k_raw, power_example = power_per_skewer(truth_matched)
power_k, _ = bin_modes(power_k_raw, power_example, 10)
bispectrum_k_raw, bispectrum_example = folded_bispectrum_per_skewer(
    truth_matched
)
bispectrum_k, _ = bin_modes(
    bispectrum_k_raw, bispectrum_example, 8
)

truth_summaries = summaries_by_volume(truth_matched)
unet_summaries = summaries_by_volume(unet_matched)
summaries_for_each_draw = [
    summaries_by_volume(sample) for sample in samples_matched
]
ddpm_summaries = {
    name: np.stack([draw[name] for draw in summaries_for_each_draw])
    for name in ("PDF", "power", "bispectrum")
}

print("Truth shapes:", {k: v.shape for k, v in truth_summaries.items()})
print("U-Net shapes:", {k: v.shape for k, v in unet_summaries.items()})
print("DDPM shapes: ", {k: v.shape for k, v in ddpm_summaries.items()})


## 14. Combine posterior uncertainty with inter-volume scatter

Let $S_{rv}$ be a summary-statistic bin from DDPM draw $r$ in held-out volume $v$. We use the law of total variance:

$$\sigma_{\rm total}^2
=\underbrace{\operatorname{Var}_v\!\left[\mathbb E_r(S_{rv}\mid v)\right]}_{\text{inter-volume variance}}
+\underbrace{\mathbb E_v\!\left[\operatorname{Var}_r(S_{rv}\mid v)\right]}_{\text{DDPM posterior variance}}.$$

This retains genuine box-to-box scatter and adds the inversion uncertainty carried by the conditional DDPM samples. It is preferable to treating the 16th–84th percentile width as an independent ad hoc error bar.


In [ ]:
def combine_ddpm_uncertainties(sample_volume_statistic):
    '''Combine sample uncertainty and inter-volume scatter bin by bin.'''
    # Shape: [posterior sample, test volume, statistic bin]
    posterior_mean_in_each_volume = np.nanmean(
        sample_volume_statistic, axis=0
    )
    central_curve = np.nanmean(posterior_mean_in_each_volume, axis=0)

    inter_volume_std = np.nanstd(
        posterior_mean_in_each_volume, axis=0, ddof=1
    )
    posterior_variance = np.nanmean(
        np.nanvar(sample_volume_statistic, axis=0, ddof=1), axis=0
    )
    total_std = np.sqrt(inter_volume_std**2 + posterior_variance)
    return central_curve, inter_volume_std, np.sqrt(posterior_variance), total_std


ddpm_combined = {
    name: combine_ddpm_uncertainties(values)
    for name, values in ddpm_summaries.items()
}

print("For each statistic we retain: mean, inter-volume sigma, "
      "posterior sigma, total sigma")
for name, components in ddpm_combined.items():
    print(name, [component.shape for component in components])


In [ ]:
coordinates = {
    "PDF": pdf_centres,
    "power": power_k,
    "bispectrum": bispectrum_k,
}
ylabels = {
    "PDF": "probability density",
    "power": r"$P_{1\rm D}(k)$ [$h^{-1}$ cMpc]",
    "bispectrum": r"$B_{1\rm D}(k,k,-2k)$ [$(h^{-1}$ cMpc$)^2$]",
}
titles = {
    "PDF": "Density PDF",
    "power": "1D power spectrum",
    "bispectrum": "Folded 1D bispectrum",
}

fig, axes = plt.subplots(3, 1, figsize=(6.3, 15.0), constrained_layout=True)

for ax, name in zip(axes, ("PDF", "power", "bispectrum")):
    coordinate = coordinates[name]
    truth_mean = np.nanmean(truth_summaries[name], axis=0)
    truth_std = np.nanstd(truth_summaries[name], axis=0, ddof=1)
    unet_mean = np.nanmean(unet_summaries[name], axis=0)
    unet_std = np.nanstd(unet_summaries[name], axis=0, ddof=1)
    ddpm_mean, volume_std, posterior_std, total_std = ddpm_combined[name]

    valid_truth = np.isfinite(coordinate) & np.isfinite(truth_mean)
    valid_unet = np.isfinite(coordinate) & np.isfinite(unet_mean)
    valid_ddpm = np.isfinite(coordinate) & np.isfinite(ddpm_mean)
    if name in ("PDF", "power"):
        valid_truth &= truth_mean > 0.0
        valid_unet &= unet_mean > 0.0
        valid_ddpm &= ddpm_mean > 0.0

    ax.errorbar(
        coordinate[valid_truth], truth_mean[valid_truth],
        yerr=truth_std[valid_truth], color=BLACK, marker="o",
        ms=3.5, lw=1.3, capsize=2, label="truth: inter-volume sigma"
    )
    ax.errorbar(
        coordinate[valid_unet], unet_mean[valid_unet],
        yerr=unet_std[valid_unet], color=RED, marker="s",
        ms=3.5, lw=1.3, capsize=2,
        label="saved U-Net: inter-volume sigma"
    )
    ax.errorbar(
        coordinate[valid_ddpm], ddpm_mean[valid_ddpm],
        yerr=total_std[valid_ddpm], color=ORANGE, marker="^",
        ms=4.0, lw=1.3, capsize=2,
        label="DDPM: total sigma"
    )
    ax.set(ylabel=ylabels[name], title=titles[name])
    ax.grid(alpha=0.18, which="both")

axes[0].set_xlabel(r"$\log_{10}\Delta_b$")
axes[1].set_xlabel(r"$k$ [$h$ cMpc$^{-1}$]")
axes[2].set_xlabel(r"$k$ [$h$ cMpc$^{-1}$]")
axes[0].set_yscale("log")
axes[1].set(xscale="log", yscale="log")
axes[2].set_xscale("log")
axes[2].axhline(0.0, color=GREY, lw=0.7)
axes[0].legend(fontsize=8)
fig.suptitle(
    "Saved U-Net and DDPM summary statistics",
    fontsize=14
)
plt.show()


## 15. Interpretation and limitations

- The median is a useful single reconstruction, but the posterior samples are the main DDPM output.
- A wider 68% band means that the observed flux permits a wider range of densities; it is not simply observational noise attached after prediction.
- The orange DDPM error bars combine inter-volume scatter with inversion uncertainty; the red saved-U-Net error bars show inter-volume scatter only.
- Pixel-wise intervals are marginal intervals. They do not display the spatial covariance that is present in the full sampled sightlines.
- The 68% coverage should be checked, not assumed. If it differs substantially from 68%, improve training/model capacity or calibrate on a separate calibration split.
- With only 32 posterior samples, estimated tails and bispectrum uncertainties are noisy. Increase `posterior_samples` for a scientific analysis.
- The forward model omits peculiar velocities, continuum uncertainty, metal contamination, and spatially varying noise. These must be added before applying the method to observed spectra.
